# Token Classification 核心总结

本笔记是 `section-02.ipynb` 的精华提炼，聚焦于 **命名实体识别（NER）** 任务的完整流程。

## 任务定义
Token Classification = 给序列中**每一个 token** 打一个标签。

代表任务：
- **NER (命名实体识别)**：识别文本中的人名、组织、地点等实体
- **POS tagging (词性标注)**：给每个词标注词性（名词、动词等）
- **Chunking (短语识别)**：识别名词短语、动词短语等

## 核心挑战
预训练模型（BERT等）使用 **subword tokenization**（子词分词），一个单词可能被拆成多个 token。
因此需要解决：**原始词级标签 → subword token 级标签** 的对齐问题。

## 完整流程图
```
原始数据集 (词+标签)
    ↓
Tokenizer (词 → subword tokens)
    ↓
标签对齐 (align_labels_with_tokens)
    ↓
DataCollator (动态 padding)
    ↓
AutoModelForTokenClassification
    ↓
训练 (Trainer API 或 Accelerate)
    ↓
推理 (pipeline)
```

In [ ]:
# 安装依赖
# !pip install datasets evaluate transformers[sentencepiece] accelerate seqeval

---
## 第一步：认识数据集与 BIO 标注体系

In [ ]:
from datasets import load_dataset

# CoNLL-2003 是经典 NER 基准数据集，包含英文新闻文本
raw_datasets = load_dataset("BramVanroy/conll2003")
raw_datasets

In [ ]:
# 查看一条训练样本：tokens 是已经分好词的单词列表，ner_tags 是对应的标签
sample = raw_datasets["train"][0]
print("tokens  :", sample["tokens"])
print("ner_tags:", sample["ner_tags"])

In [ ]:
# ===============================================================
# BIO 标注体系 (Begin-Inside-Outside)
# ===============================================================
# B-XXX : 实体 XXX 的开头 token
# I-XXX : 实体 XXX 的中间/结尾 token（紧跟 B-XXX 之后）
# O     : 不属于任何实体（Outside）
#
# 例："Hugging Face" 是一个 ORG 实体
#   "Hugging" -> B-ORG
#   "Face"    -> I-ORG
#
# 为什么需要 B/I 区分？
# 若两个同类实体相邻，如 "Apple Microsoft"，
# 用 B-ORG I-ORG 就能区分为两个独立实体，而不是一个。

# 获取标签名称列表（从 ClassLabel 特征中提取）
ner_feature = raw_datasets["train"].features["ner_tags"]
label_names = ner_feature.feature.names
print("标签列表:", label_names)
# ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

In [ ]:
# 可视化：对齐显示 word 和对应的 NER 标签
words = raw_datasets["train"][4]["tokens"]
labels = raw_datasets["train"][4]["ner_tags"]

line1, line2 = "", ""
for word, label in zip(words, labels):
    full_label = label_names[label]
    width = max(len(word), len(full_label)) + 1
    line1 += word.ljust(width)
    line2 += full_label.ljust(width)

print(line1)
print(line2)

---
## 第二步（核心难点）：Subword Tokenization 与标签对齐

### 问题所在
BERT 使用 WordPiece 分词，一个单词可能被切成多个 subword token：

```
原始词:      Zwingmann
分词结果:    Z  ##wing  ##mann    ← 一个词变成了3个token！
原始标签:    B-PER
需要的标签:  B-PER  I-PER  I-PER  ← 需要为每个subword分配标签
```

### 对齐规则
- **特殊 token** (`[CLS]`, `[SEP]`)：标签设为 `-100`（PyTorch 会在计算 loss 时自动忽略）
- **词的第一个 subword**：保留原始标签
- **词的后续 subword**：
  - 若原标签是 `B-XXX`（奇数），改为 `I-XXX`（偶数 = B+1）
  - 若原标签是 `I-XXX` 或 `O`，保持不变

In [ ]:
from transformers import AutoTokenizer

model_checkpoint = "bert-base-cased"
# 使用 Fast Tokenizer（基于 Rust 实现），速度快且提供 word_ids() 功能
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

print("is_fast:", tokenizer.is_fast)  # 必须为 True，否则 word_ids() 不可用

In [ ]:
# is_split_into_words=True 告诉 tokenizer 输入已经是词列表，不需要再做分词
inputs = tokenizer(raw_datasets["train"][4]["tokens"], is_split_into_words=True)

print("subword tokens:", inputs.tokens())
print()

# word_ids() 是关键：返回每个 subword token 对应的原始词索引
# None 表示特殊 token ([CLS], [SEP])
word_ids = inputs.word_ids()
print("word_ids       :", word_ids)
# 例如: [None, 0, 1, 1, 2, 3, ...]
# 索引 1 和 2 的 token 都属于原始词 1，说明该词被拆成了 2 个 subword

In [ ]:
def align_labels_with_tokens(labels, word_ids):
    """
    将词级别的 NER 标签对齐到 subword token 级别。
    
    参数：
        labels   : 原始词级标签列表，如 [5, 0, 0, 3, 4, ...]
        word_ids : tokenizer 输出的 word_ids()，如 [None, 0, 1, 1, 2, ...]
    
    返回：
        new_labels : subword token 级别的标签列表
    """
    new_labels = []
    current_word = None  # 追踪当前处理的是哪个词
    
    for word_id in word_ids:
        if word_id is None:
            # 特殊 token（[CLS] / [SEP]），标签设为 -100
            # -100 是 PyTorch CrossEntropyLoss 的 ignore_index，不参与 loss 计算
            new_labels.append(-100)
        elif word_id != current_word:
            # 遇到新词的第一个 subword → 使用原始词的标签
            current_word = word_id
            new_labels.append(labels[word_id])
        else:
            # 同一个词的后续 subword
            label = labels[word_id]
            # 若是 B-XXX（奇数），后续 subword 应标为 I-XXX
            # BIO 标签的编码规律：B-PER=1, I-PER=2, B-ORG=3, I-ORG=4 ...
            # 即 B 标签都是奇数，对应的 I 标签 = B + 1
            if label % 2 == 1:
                label += 1  # B-XXX → I-XXX
            new_labels.append(label)
    
    return new_labels

In [ ]:
# 验证对齐效果
labels = raw_datasets["train"][4]["ner_tags"]
word_ids = inputs.word_ids()

print("word_ids      :", word_ids)
print("原始词标签     :", labels)
print("对齐后token标签:", align_labels_with_tokens(labels, word_ids))

# 可以看到：
# - 首尾的 None 变成了 -100
# - 同一个词的多个 subword 共享标签
# - B-XXX 在后续 subword 中变成了 I-XXX

In [ ]:
def tokenize_and_align_labels(examples):
    """
    批量处理：对整批样本完成分词 + 标签对齐。
    用于 dataset.map() 的批处理函数。
    """
    # 批量分词（truncation=True 截断超长序列）
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,  # 输入已经是词列表
    )
    
    all_labels = examples["ner_tags"]
    new_labels = []
    for i, labels in enumerate(all_labels):
        word_ids = tokenized_inputs.word_ids(i)  # 第 i 条样本的 word_ids
        new_labels.append(align_labels_with_tokens(labels, word_ids))
    
    tokenized_inputs["labels"] = new_labels
    return tokenized_inputs


# 对全数据集应用，batched=True 加速处理
tokenized_datasets = raw_datasets.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,  # 移除原始列，只保留模型需要的
)
tokenized_datasets

---
## 第三步：Data Collator（动态 Padding）

不同样本的 token 序列长度不同，批训练时需要 padding 到相同长度。

`DataCollatorForTokenClassification` 的特别之处：
- 不仅对 `input_ids` 做 padding
- 同时对 **`labels`** 也做 padding，padding 值为 **`-100`**（不参与 loss）

In [ ]:
from transformers import DataCollatorForTokenClassification

# DataCollatorForTokenClassification 与 DataCollatorWithPadding 的区别：
# 后者只 pad input_ids/attention_mask，
# 前者额外处理 labels 的 padding（用 -100 填充）
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# 演示：对两条长度不同的样本做 collate
batch = data_collator([tokenized_datasets["train"][i] for i in range(2)])
print("样本0原始标签长度:", len(tokenized_datasets["train"][0]["labels"]))
print("样本1原始标签长度:", len(tokenized_datasets["train"][1]["labels"]))
print()
print("Batch 后的 labels（短的被 -100 填充到与长的一致）:")
print(batch["labels"])

---
## 第四步：评估指标 — seqeval

NER 任务使用 **seqeval** 库进行评估，而不是简单的 accuracy。

原因：NER 关注的是**实体级别**的识别是否正确，而不是单个 token。
例如「Hugging Face」是一个实体，必须两个词都对才算识别正确。

关键指标：
- **Precision**（精确率）：预测为实体中，真正是实体的比例
- **Recall**（召回率）：真实实体中，被成功预测到的比例
- **F1**：Precision 和 Recall 的调和平均，综合评估指标

In [ ]:
import evaluate
import numpy as np

metric = evaluate.load("seqeval")

# seqeval 要求输入是字符串标签列表，而不是数字
example_labels = [label_names[i] for i in raw_datasets["train"][0]["ner_tags"]]
example_preds = example_labels.copy()
example_preds[2] = "O"  # 故意弄错一个预测

result = metric.compute(predictions=[example_preds], references=[example_labels])
print(result)

In [ ]:
def compute_metrics(eval_preds):
    """
    供 Trainer 调用的评估函数。
    
    eval_preds 是一个 (logits, labels) 元组：
    - logits: shape (batch_size, seq_len, num_labels) 的 numpy 数组
    - labels: shape (batch_size, seq_len) 的 numpy 数组，-100 为忽略位
    """
    logits, labels = eval_preds
    
    # argmax 得到每个 token 的预测类别
    predictions = np.argmax(logits, axis=-1)
    
    # 过滤掉 -100 位置（特殊 token），将数字 id 转换为字符串标签
    true_labels = [
        [label_names[l] for l in label if l != -100]
        for label in labels
    ]
    true_predictions = [
        [label_names[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    
    all_metrics = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": all_metrics["overall_precision"],
        "recall":    all_metrics["overall_recall"],
        "f1":        all_metrics["overall_f1"],
        "accuracy":  all_metrics["overall_accuracy"],
    }

---
## 第五步：加载模型

`AutoModelForTokenClassification` 的结构：
```
BertModel (预训练编码器，输出每个 token 的 768 维向量)
    ↓
Linear(768 → num_labels)  ← 新初始化的分类头
    ↓
每个 token 对应一个 num_labels 维的 logit 向量
```

In [ ]:
from transformers import AutoModelForTokenClassification

# id2label 和 label2id 存储到 model.config，推理时用于将预测 id 转回标签字符串
id2label = {i: label for i, label in enumerate(label_names)}
label2id = {v: k for k, v in id2label.items()}

model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    id2label=id2label,
    label2id=label2id,
)

# 加载时会看到警告：
# - UNEXPECTED: bert-base-cased 原本有 MLM 头（cls.predictions.*），这里不需要
# - MISSING: 新的分类头 classifier.weight/bias 会随机初始化
# 这是正常的 fine-tuning 过程

print("分类标签数量:", model.config.num_labels)  # 应为 9

---
## 第六步（方案A）：使用 Trainer API 训练

适合快速实验，高度封装，几行代码完成训练。

In [ ]:
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir="bert-finetuned-ner",   # 模型保存目录
    push_to_hub=True,                  # 训练后推送到 HuggingFace Hub
    
    # 学习率相关
    learning_rate=2e-5,                # 微调典型学习率，比预训练小很多
    weight_decay=0.01,                 # L2 正则化
    lr_scheduler_type="linear",        # 学习率线性衰减
    warmup_ratio=0.1,                  # 前 10% 步数做 warmup
    
    # 批次大小
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    
    # 训练轮数
    num_train_epochs=3,
    
    # 评估与保存策略
    eval_strategy="epoch",             # 每个 epoch 结束后评估
    save_strategy="epoch",
    load_best_model_at_end=True,       # 训练结束时加载最佳模型
    metric_for_best_model="f1",        # 用 F1 选择最佳模型
    greater_is_better=True,
    
    fp16=True,                         # 混合精度训练，加速并节省显存
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,        # 新版 API，等价于 tokenizer=tokenizer
)

# trainer.train()
# trainer.push_to_hub(commit_message="Training complete")

---
## 第六步（方案B）：使用 Accelerate 自定义训练循环

适合需要**精细控制**训练过程的场景（自定义 loss、梯度裁剪、多 GPU 策略等）。

`Accelerate` 的核心价值：用同一份代码，无缝支持 CPU / 单 GPU / 多 GPU / TPU。

In [ ]:
from torch.utils.data import DataLoader
from torch.optim import AdamW
from accelerate import Accelerator
from transformers import get_scheduler, AutoModelForTokenClassification
from huggingface_hub import HfApi, get_full_repo_name
from tqdm.auto import tqdm
import torch

# ── 1. 准备 DataLoader ──────────────────────────────────────────
train_dataloader = DataLoader(
    tokenized_datasets["train"],
    shuffle=True,              # 训练时打乱
    collate_fn=data_collator,
    batch_size=8,
)
eval_dataloader = DataLoader(
    tokenized_datasets["validation"],
    collate_fn=data_collator,
    batch_size=8,
)

# ── 2. 重新加载模型（避免复用已训练的参数）──────────────────────
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint, id2label=id2label, label2id=label2id
)

# ── 3. Accelerator 初始化 ────────────────────────────────────────
# Accelerator 会自动检测运行环境（CPU/GPU/TPU），并配置相应的分布式策略
accelerator = Accelerator()

# ── 4. 优化器 ───────────────────────────────────────────────────
optimizer = AdamW(model.parameters(), lr=2e-5)

# ── 5. 关键：用 accelerator.prepare() 包装所有组件 ──────────────
# prepare() 会自动处理：设备迁移(.to(device))、多 GPU 并行包装(DDP)、精度转换等
model, optimizer, train_dataloader, eval_dataloader = accelerator.prepare(
    model, optimizer, train_dataloader, eval_dataloader
)

# ── 6. 学习率调度器 ─────────────────────────────────────────────
num_train_epochs = 3
num_training_steps = num_train_epochs * len(train_dataloader)

lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

In [ ]:
def postprocess(predictions, labels):
    """
    将模型输出的预测张量转换为 seqeval 需要的字符串列表格式。
    同时过滤掉 -100 位置（特殊 token）。
    """
    predictions = predictions.detach().cpu().numpy()
    labels = labels.detach().cpu().numpy()
    
    true_labels = [
        [label_names[l] for l in label if l != -100]
        for label in labels
    ]
    true_predictions = [
        [label_names[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    return true_labels, true_predictions

In [ ]:
# ── Hub 准备 ────────────────────────────────────────────────────
api = HfApi()
model_name = "bert-finetuned-ner-accelerate"
repo_name = get_full_repo_name(model_name)
output_dir = model_name
# api.create_repo(repo_name, exist_ok=True)

# ── 训练主循环 ──────────────────────────────────────────────────
progress_bar = tqdm(range(num_training_steps))

for epoch in range(num_train_epochs):
    
    # ===== 训练阶段 =====
    model.train()
    for batch in train_dataloader:
        # batch 已经由 accelerator 自动迁移到正确设备
        outputs = model(**batch)      # 前向传播，自动计算 loss
        loss = outputs.loss
        
        # accelerator.backward() 替代 loss.backward()
        # 在混合精度/分布式场景下会做额外处理（梯度缩放等）
        accelerator.backward(loss)
        
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)
    
    # ===== 评估阶段 =====
    model.eval()
    for batch in eval_dataloader:
        with torch.no_grad():
            outputs = model(**batch)
        
        predictions = outputs.logits.argmax(dim=-1)
        labels = batch["labels"]
        
        # 多 GPU 场景：各 GPU 的序列长度可能不同，需要 pad 后才能 gather
        predictions = accelerator.pad_across_processes(predictions, dim=1, pad_index=-100)
        labels      = accelerator.pad_across_processes(labels,      dim=1, pad_index=-100)
        
        # gather 将所有 GPU 的结果汇总到主进程
        predictions_gathered = accelerator.gather(predictions)
        labels_gathered      = accelerator.gather(labels)
        
        true_labels, true_predictions = postprocess(predictions_gathered, labels_gathered)
        metric.add_batch(predictions=true_predictions, references=true_labels)
    
    results = metric.compute()
    print(f"Epoch {epoch}:", {k: results[f"overall_{k}"] for k in ["precision", "recall", "f1", "accuracy"]})
    
    # ===== 保存模型 =====
    accelerator.wait_for_everyone()  # 确保所有进程到达同一点
    unwrapped_model = accelerator.unwrap_model(model)  # 移除 DDP 包装，恢复原始模型
    unwrapped_model.save_pretrained(output_dir, save_function=accelerator.save)
    
    if accelerator.is_main_process:  # 只在主进程中保存 tokenizer 和上传
        tokenizer.save_pretrained(output_dir)
        # api.upload_folder(
        #     folder_path=output_dir,
        #     path_in_repo=".",
        #     repo_id=repo_name,
        #     commit_message=f"Training in progress epoch {epoch}",
        # )

---
## 第七步：使用 pipeline 推理

`aggregation_strategy="simple"` 会自动将连续的 B-XXX / I-XXX token 合并为一个实体。

In [ ]:
from transformers import pipeline

# 使用官方课程提供的已训练模型
model_checkpoint = "huggingface-course/bert-finetuned-ner"
token_classifier = pipeline(
    "token-classification",
    model=model_checkpoint,
    aggregation_strategy="simple",  # 将 B/I token 合并为实体
)

result = token_classifier("My name is Sylvain and I work at Hugging Face in Brooklyn.")
for entity in result:
    print(f"  [{entity['entity_group']}] '{entity['word']}' (score: {entity['score']:.4f})")

# 输出示例：
# [PER] 'Sylvain'      (score: 0.9989)
# [ORG] 'Hugging Face' (score: 0.9648)
# [LOC] 'Brooklyn'     (score: 0.9986)

---
## 总结：核心知识点速查

### 1. BIO 标注体系
| 标签 | 含义 | 编码规律 |
|------|------|----------|
| O | 非实体 | 0 |
| B-PER | 人名开头 | 奇数 (1) |
| I-PER | 人名内部 | 偶数 (2) |
| B-ORG | 机构开头 | 奇数 (3) |
| I-ORG | 机构内部 | 偶数 (4) |

### 2. 标签对齐的三条规则
```
word_id == None    → label = -100    # 特殊 token，忽略
第一个 subword     → label = 原始标签
后续 subword       → label = 原始标签（若是 B-XXX 则改为 I-XXX）
```

### 3. -100 的作用
PyTorch `CrossEntropyLoss` 的 `ignore_index=-100`：
标签为 -100 的位置不参与 loss 计算，也不参与评估。

### 4. 两种训练方式对比
| 方式 | 优点 | 缺点 |
|------|------|------|
| Trainer API | 代码少，自动处理多 GPU、混合精度、checkpoint | 灵活性低 |
| Accelerate | 完全自定义训练逻辑，分布式透明 | 需要手动写训练循环 |

### 5. `word_ids()` 的重要性
`word_ids()` 是 Fast Tokenizer 提供的关键功能，返回每个 subword token 对应的原始词索引，是实现标签对齐的基础。

### 6. 模型架构
```
输入: [CLS] token1 subword21 subword22 token3 [SEP]
         ↓
BERT Encoder (12层 Transformer)
         ↓ 每个 token 得到 768 维向量
Linear(768 → 9)  # 9 = num_labels
         ↓
logits: shape (seq_len, 9)
         ↓ argmax
每个 token 的预测标签
```